# Mathula TV OpenVoice GPU worker (Kaggle)

Run the numbered stages from top to bottom in a private GPU runtime. Set `MATHULA_TV_OPENVOICE_WORKER_MODE=assets` for GPU asset preflight or `conversion` for queued turn conversion. The server-prepared, self-verifying plan is authoritative. This notebook contains orchestration glue only: embedding extraction, conversion, validation, resumability, leases, manifests, and promotion use the installed `mathula_tv` package. Configure immutable repository/OpenVoice commit SHAs, exact Python/Torch versions, checkpoint SHA-256 values, the selected plan SHA-256, a zero-argument `MATHULA_TV_OPENVOICE_RUNTIME_FACTORY`, and the explicit live-operation acknowledgement in the runtime environment. Saved outputs must remain empty.

## 01. Runtime inspection

In [ ]:
import hashlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path, PurePosixPath
from urllib.parse import urlsplit

PLATFORM = "kaggle"
SHA256_RE = re.compile(r"[0-9a-f]{64}")
COMMIT_RE = re.compile(r"[0-9a-f]{40}")
SAFE_ID_RE = re.compile(r"[A-Za-z0-9][A-Za-z0-9_.-]{0,127}")
EXPLICIT_ACK = "I_ACKNOWLEDGE_LIVE_OPENVOICE_GCS_MUTATION"

def required_env(name):
    value = os.environ.get(name, "").strip()
    if not value:
        raise RuntimeError(f"Required worker configuration is missing: {name}")
    return value

def public_repository(name):
    value = required_env(name)
    parsed = urlsplit(value)
    if parsed.scheme != "https" or not parsed.hostname or parsed.username or parsed.password or parsed.query or parsed.fragment:
        raise ValueError(f"{name} must be a public HTTPS repository URL without credentials or query parameters")
    return value

def run_quiet(arguments, label):
    completed = subprocess.run(arguments, text=True, capture_output=True, check=False)
    if completed.returncode:
        raise RuntimeError(f"{label} failed with exit code {completed.returncode}")
    return completed.stdout.strip()

JOB_ID = required_env("MATHULA_TV_JOB_ID")
WORKER_ID = required_env("MATHULA_TV_WORKER_ID")
if not SAFE_ID_RE.fullmatch(JOB_ID) or not SAFE_ID_RE.fullmatch(WORKER_ID):
    raise ValueError("Job and worker identifiers must use safe path characters")
GCP_PROJECT = required_env("MATHULA_TV_GCP_PROJECT")
GCS_BUCKET = required_env("MATHULA_TV_GCS_BUCKET")
GCS_PREFIX = os.environ.get("MATHULA_TV_GCS_PREFIX", "mathula-tv").strip("/")
REPOSITORY_URL = public_repository("MATHULA_TV_REPOSITORY_URL")
REPOSITORY_COMMIT = required_env("MATHULA_TV_REPOSITORY_COMMIT")
OPENVOICE_REPOSITORY_URL = public_repository("MATHULA_TV_OPENVOICE_REPOSITORY_URL")
OPENVOICE_REVISION = required_env("MATHULA_TV_OPENVOICE_REVISION")
WORKER_MODE = os.environ.get("MATHULA_TV_OPENVOICE_WORKER_MODE", "").strip().lower()
if WORKER_MODE not in {"assets", "conversion"}:
    raise ValueError("MATHULA_TV_OPENVOICE_WORKER_MODE must be assets or conversion")
PLAN_SHA_ENV = "MATHULA_TV_OPENVOICE_ASSET_PLAN_SHA256" if WORKER_MODE == "assets" else "MATHULA_TV_OPENVOICE_PLAN_SHA256"
PLAN_SHA256 = required_env(PLAN_SHA_ENV)
PLAN_RELATIVE = "dubbing/openvoice/asset_plan.json" if WORKER_MODE == "assets" else "dubbing/openvoice/plan.json"
MANIFEST_RELATIVE = "dubbing/openvoice/asset_manifest.json" if WORKER_MODE == "assets" else "dubbing/openvoice/manifest.json"
LEASE_TASK = "openvoice_asset_preparation" if WORKER_MODE == "assets" else "openvoice_conversion"
CANDIDATE_EVALUATOR_FACTORY = os.environ.get("MATHULA_TV_OPENVOICE_CANDIDATE_EVALUATOR_FACTORY", "").strip()
EXPECTED_PYTHON_VERSION = required_env("MATHULA_TV_OPENVOICE_PYTHON_VERSION")
EXPECTED_TORCH_VERSION = required_env("MATHULA_TV_OPENVOICE_TORCH_VERSION")
RUNTIME_FACTORY = required_env("MATHULA_TV_OPENVOICE_RUNTIME_FACTORY")
LIVE_OPERATION = os.environ.get("MATHULA_TV_LIVE_OPERATION", "0")
LIVE_OPERATION_ACK = os.environ.get("MATHULA_TV_LIVE_OPERATION_ACK", "")
LEASE_SECONDS = int(os.environ.get("MATHULA_TV_LEASE_SECONDS", "900"))
MAX_DOWNLOAD_BYTES = int(os.environ.get("MATHULA_TV_MAX_ARTIFACT_DOWNLOAD_BYTES", str(2 * 1024 * 1024 * 1024)))
MAX_ATTEMPTS = int(os.environ.get("MATHULA_TV_OPENVOICE_MAX_ATTEMPTS", "2"))
PRECISION = os.environ.get("MATHULA_TV_OPENVOICE_PRECISION", "fp16")
if not COMMIT_RE.fullmatch(REPOSITORY_COMMIT) or not COMMIT_RE.fullmatch(OPENVOICE_REVISION):
    raise ValueError("Repository and OpenVoice revisions must be immutable 40-character commit SHAs")
if not SHA256_RE.fullmatch(PLAN_SHA256):
    raise ValueError("The server-provided OpenVoice worker plan SHA-256 is required")
if LEASE_SECONDS < 60 or MAX_DOWNLOAD_BYTES < 1 or MAX_ATTEMPTS < 1 or PRECISION not in {"fp16", "fp32", "bf16"}:
    raise ValueError("Invalid bounded worker configuration")
WORKER_REQUIREMENTS = json.loads(required_env("MATHULA_TV_OPENVOICE_REQUIREMENTS_JSON"))
CHECKPOINT_SPECS = json.loads(required_env("MATHULA_TV_OPENVOICE_CHECKPOINTS_JSON"))
EXACT_REQUIREMENT_RE = re.compile(r"[A-Za-z0-9][A-Za-z0-9_.-]*(?:\[[A-Za-z0-9_,.-]+\])?==[A-Za-z0-9][A-Za-z0-9_.+!-]*")
if not isinstance(WORKER_REQUIREMENTS, list) or not WORKER_REQUIREMENTS or any(not isinstance(item, str) or not EXACT_REQUIREMENT_RE.fullmatch(item) for item in WORKER_REQUIREMENTS):
    raise ValueError("Worker dependencies must be a non-empty JSON list of exact == pins")
if not isinstance(CHECKPOINT_SPECS, list) or not CHECKPOINT_SPECS:
    raise ValueError("Checkpoint configuration must be a non-empty JSON list")
WORK_ROOT = Path("/kaggle/working/mathula-tv-openvoice-worker").resolve()
REPOSITORY_ROOT = WORK_ROOT / "repository"
OPENVOICE_ROOT = WORK_ROOT / "openvoice"
CHECKPOINT_ROOT = WORK_ROOT / "checkpoints"
JOB_ROOT = WORK_ROOT / "jobs" / JOB_ID
WORK_ROOT.mkdir(parents=True, exist_ok=True)
gpu_query = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True, capture_output=True, check=False)
gpu_name = gpu_query.stdout.strip().splitlines()[0] if gpu_query.returncode == 0 and gpu_query.stdout.strip() else "unavailable"
print({"runtime_platform": PLATFORM, "worker_mode": WORKER_MODE, "job_id": JOB_ID, "gpu": gpu_name, "python_version": sys.version.split()[0]})

## 02. Python and CUDA compatibility

In [ ]:
import torch

python_version = sys.version.split()[0]
torch_version = torch.__version__
if python_version != EXPECTED_PYTHON_VERSION or torch_version != EXPECTED_TORCH_VERSION:
    raise RuntimeError("The GPU runtime does not match the reviewed Python/Torch pins")
if not torch.cuda.is_available() or not torch.version.cuda:
    raise RuntimeError("Select a CUDA-enabled Kaggle GPU runtime")
gpu_name = torch.cuda.get_device_name(0)
print({"python_version": python_version, "torch_version": torch_version, "cuda_version": torch.version.cuda, "gpu": gpu_name})

## 03. Repository checkout

In [ ]:
if REPOSITORY_ROOT.exists():
    if not (REPOSITORY_ROOT / ".git").is_dir():
        raise RuntimeError("The repository work path exists but is not a Git checkout")
else:
    run_quiet(["git", "clone", "--filter=blob:none", "--no-checkout", REPOSITORY_URL, str(REPOSITORY_ROOT)], "Mathula TV repository clone")
run_quiet(["git", "-C", str(REPOSITORY_ROOT), "checkout", "--detach", REPOSITORY_COMMIT], "Mathula TV repository checkout")
checked_out_commit = run_quiet(["git", "-C", str(REPOSITORY_ROOT), "rev-parse", "HEAD"], "Mathula TV revision inspection")
if checked_out_commit != REPOSITORY_COMMIT:
    raise RuntimeError("Mathula TV checkout did not resolve to the reviewed commit")
print({"repository_commit": checked_out_commit})

## 04. Dependency installation

In [ ]:
run_quiet([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-input", *WORKER_REQUIREMENTS], "Pinned worker dependency installation")
run_quiet([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-input", "--no-deps", str(REPOSITORY_ROOT)], "Pinned Mathula TV installation")
print({"dependencies_installed": len(WORKER_REQUIREMENTS)})

## 05. OpenVoice revision pinning

In [ ]:
if OPENVOICE_ROOT.exists():
    if not (OPENVOICE_ROOT / ".git").is_dir():
        raise RuntimeError("The OpenVoice work path exists but is not a Git checkout")
else:
    run_quiet(["git", "clone", "--filter=blob:none", "--no-checkout", OPENVOICE_REPOSITORY_URL, str(OPENVOICE_ROOT)], "OpenVoice repository clone")
run_quiet(["git", "-C", str(OPENVOICE_ROOT), "checkout", "--detach", OPENVOICE_REVISION], "OpenVoice revision checkout")
checked_out_openvoice = run_quiet(["git", "-C", str(OPENVOICE_ROOT), "rev-parse", "HEAD"], "OpenVoice revision inspection")
if checked_out_openvoice != OPENVOICE_REVISION:
    raise RuntimeError("OpenVoice checkout did not resolve to the reviewed commit")
run_quiet([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-input", "--no-deps", str(OPENVOICE_ROOT)], "Pinned OpenVoice installation")
os.environ["MATHULA_TV_OPENVOICE_CHECKOUT"] = str(OPENVOICE_ROOT)
print({"openvoice_revision": checked_out_openvoice})

## 06. Checkpoint materialisation

In [ ]:
import shutil

CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
materialized_checkpoints = {}
for spec in CHECKPOINT_SPECS:
    if not isinstance(spec, dict):
        raise ValueError("Every checkpoint specification must be an object")
    name = str(spec.get("name", ""))
    filename = str(spec.get("filename", ""))
    expected_hash = str(spec.get("sha256", ""))
    if not SAFE_ID_RE.fullmatch(name) or Path(filename).name != filename or not filename or not SHA256_RE.fullmatch(expected_hash):
        raise ValueError("Checkpoint names, filenames, and SHA-256 values must be explicit and safe")
    source = Path(str(spec.get("source_path", ""))).expanduser().resolve()
    if not source.is_file():
        raise FileNotFoundError(f"A configured checkpoint source is unavailable: {name}")
    destination = CHECKPOINT_ROOT / filename
    shutil.copyfile(source, destination)
    if name in materialized_checkpoints:
        raise ValueError("Checkpoint names must be unique")
    materialized_checkpoints[name] = (destination, expected_hash)
print({"checkpoint_count": len(materialized_checkpoints)})

## 07. Checkpoint hash verification

In [ ]:
from mathula_tv.openvoice import CheckpointAsset, sha256_file

checkpoint_assets = []
checkpoint_hashes = {}
for name, (path, expected_hash) in sorted(materialized_checkpoints.items()):
    actual_hash = sha256_file(path)
    if actual_hash != expected_hash:
        raise RuntimeError(f"Pinned checkpoint hash mismatch: {name}")
    checkpoint_assets.append(CheckpointAsset(name=name, path=path, sha256=actual_hash))
    checkpoint_hashes[name] = actual_hash
print({"checkpoint_hashes": checkpoint_hashes})

## 08. Authentication

In [ ]:
from google.cloud import storage
from google.oauth2 import service_account
from kaggle_secrets import UserSecretsClient

credential_text = UserSecretsClient().get_secret("MATHULA_TV_GCP_SERVICE_ACCOUNT_JSON")
if not credential_text:
    raise RuntimeError("Configure the Kaggle GCP service-account secret before authentication")
credential_payload = json.loads(credential_text)
kaggle_credentials = service_account.Credentials.from_service_account_info(credential_payload)
credential_payload.clear()
credential_payload = None
credential_text = None
gcs_client = storage.Client(project=GCP_PROJECT, credentials=kaggle_credentials)
print({"authentication": "configured"})

## 09. GCS connectivity

In [ ]:
from mathula_tv.gcs_store import GCSLeaseHeartbeat, GCSStore

store = GCSStore(GCS_BUCKET, GCS_PREFIX, client=gcs_client)
if not store.bucket.exists():
    raise RuntimeError("The configured GCS bucket is unavailable to this runtime")
print({"gcs_connectivity": "verified"})

## 10. Job discovery

In [ ]:
status, status_generation = store.download_json(JOB_ID, "status.json")
allowed_states = {"azure_tts_ready"} if WORKER_MODE == "assets" else {"voice_conversion_queued", "voice_conversion_running"}
if status.get("job_id") != JOB_ID or status.get("state") not in allowed_states:
    raise RuntimeError("The selected job is not eligible for the configured OpenVoice worker mode")
plan_blob = store.bucket.blob(store.name(JOB_ID, PLAN_RELATIVE))
if not plan_blob.exists():
    raise FileNotFoundError("The server-prepared OpenVoice worker plan is missing")
job_state = str(status["state"])
print({"worker_mode": WORKER_MODE, "job_id": JOB_ID, "state": job_state})

## 11. Lease claim

In [ ]:
if LIVE_OPERATION != "1" or LIVE_OPERATION_ACK != EXPLICIT_ACK:
    raise RuntimeError("Set the live-operation flag and exact acknowledgement before claiming the production lease")
attempt = int(os.environ.get("MATHULA_TV_OPENVOICE_ATTEMPT", str(int(status.get("attempt_counters", {}).get(LEASE_TASK, 0)) + 1)))
if attempt < 1:
    raise ValueError("OpenVoice attempt must be positive")
claim = store.claim(JOB_ID, LEASE_TASK, WORKER_ID, attempt, LEASE_SECONDS)
print({"worker_mode": WORKER_MODE, "job_id": JOB_ID, "lease_owner": claim["worker_id"], "lease_expiry": claim["lease_expiry"], "attempt": claim["attempt_number"]})

## 12. Plan download

In [ ]:
from mathula_tv.openvoice_worker import OpenVoiceAssetPlan, OpenVoicePlan

plan_path = JOB_ROOT / PLAN_RELATIVE
store.download(JOB_ID, PLAN_RELATIVE, plan_path)
if sha256_file(plan_path) != PLAN_SHA256:
    raise RuntimeError("Downloaded OpenVoice worker plan SHA-256 does not match the server handoff")
plan_payload = json.loads(plan_path.read_text(encoding="utf-8"))
if plan_payload.get("job_id") != JOB_ID or plan_payload.get("openvoice_revision") != OPENVOICE_REVISION or plan_payload.get("lease_identity") != "unbound":
    raise RuntimeError("OpenVoice worker plan identity, revision, or lease binding is invalid")

def job_relative(value):
    text = str(value)
    relative = PurePosixPath(text)
    if not text or text.startswith("/") or "\\" in text or any(not SAFE_ID_RE.fullmatch(part) for part in relative.parts):
        raise ValueError("OpenVoice plans may contain only job-relative POSIX artifact paths")
    return relative.as_posix()

remote_units = {}
remote_sources = {}
remote_speakers = {}
speaker_reference_specs = {}
if WORKER_MODE == "assets":
    OpenVoiceAssetPlan.from_dict(plan_payload)
    local_payload = dict(plan_payload)
    local_payload["lease_identity"] = WORKER_ID
    local_sources = []
    for raw_source in plan_payload.get("sources", []):
        voice = str(raw_source.get("azure_voice", ""))
        if not SAFE_ID_RE.fullmatch(voice) or voice in remote_sources:
            raise ValueError("Asset-plan Azure voices must be unique and safe")
        remote = {}
        local = dict(raw_source)
        for field in ("calibration_audio_path", "calibration_manifest_path", "source_embedding_path", "source_embedding_manifest_path"):
            remote[field] = job_relative(raw_source[field])
            local[field] = str(JOB_ROOT / remote[field])
        remote_sources[voice] = remote
        local_sources.append(local)
    local_speakers = []
    for raw_speaker in plan_payload.get("speakers", []):
        speaker_id = str(raw_speaker.get("speaker_id", ""))
        if not SAFE_ID_RE.fullmatch(speaker_id) or speaker_id in remote_speakers:
            raise ValueError("Asset-plan speaker IDs must be unique and safe")
        remote = {"voice_candidates": {}}
        local = dict(raw_speaker)
        for field in ("reference_audio_path", "reference_manifest_path", "target_embedding_path", "target_embedding_manifest_path"):
            remote[field] = job_relative(raw_speaker[field])
            local[field] = str(JOB_ROOT / remote[field])
        candidate_manifest = raw_speaker.get("candidate_manifest_path")
        if candidate_manifest is not None:
            remote["candidate_manifest_path"] = job_relative(candidate_manifest)
            local["candidate_manifest_path"] = str(JOB_ROOT / remote["candidate_manifest_path"])
        local_candidates = []
        for raw_candidate in raw_speaker.get("voice_candidates", []):
            candidate_id = str(raw_candidate.get("candidate_id", ""))
            if not SAFE_ID_RE.fullmatch(candidate_id) or candidate_id in remote["voice_candidates"]:
                raise ValueError("Asset-plan candidate IDs must be unique and safe per speaker")
            candidate_remote = {}
            candidate_local = dict(raw_candidate)
            for field in ("source_audio_path", "output_path"):
                candidate_remote[field] = job_relative(raw_candidate[field])
                candidate_local[field] = str(JOB_ROOT / candidate_remote[field])
            remote["voice_candidates"][candidate_id] = candidate_remote
            local_candidates.append(candidate_local)
        local["voice_candidates"] = local_candidates
        remote_speakers[speaker_id] = remote
        local_speakers.append(local)
    local_payload["sources"] = local_sources
    local_payload["speakers"] = local_speakers
    local_payload_without_hash = {key: value for key, value in local_payload.items() if key != "artifact_sha256"}
    local_payload["artifact_sha256"] = hashlib.sha256(json.dumps(local_payload_without_hash, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")).hexdigest()
    plan = OpenVoiceAssetPlan.from_dict(local_payload)
    speakers = [speaker.speaker_id for speaker in plan.speakers]
    number_of_assets = len(plan.sources) + len(plan.speakers) + sum(len(speaker.voice_candidates) for speaker in plan.speakers)
    print({"worker_mode": WORKER_MODE, "job_id": JOB_ID, "plan_sha256": PLAN_SHA256, "number_of_speakers": len(speakers), "number_of_assets": number_of_assets})
else:
    local_payload = dict(plan_payload)
    local_payload["lease_identity"] = WORKER_ID
    local_units = []
    for raw_unit in plan_payload.get("units", []):
        unit_id = str(raw_unit.get("unit_id", ""))
        if not SAFE_ID_RE.fullmatch(unit_id) or unit_id in remote_units:
            raise ValueError("OpenVoice plan unit IDs must be unique and safe")
        remote = {}
        local = dict(raw_unit)
        for field in ("source_audio_path", "source_embedding_path", "target_embedding_path", "output_path"):
            remote[field] = job_relative(raw_unit[field])
            local[field] = str(JOB_ROOT / remote[field])
        remote_units[unit_id] = remote
        local_units.append(local)
    local_payload["units"] = local_units
    plan = OpenVoicePlan.from_dict(local_payload)
    speaker_reference_specs = plan_payload.get("speaker_references", {})
    speakers = sorted({unit.speaker_id for unit in plan.units})
    if set(speaker_reference_specs) != set(speakers):
        raise RuntimeError("The plan must contain one same-speaker reference reel per job-local speaker")
    print({"worker_mode": WORKER_MODE, "job_id": JOB_ID, "plan_sha256": PLAN_SHA256, "number_of_speakers": len(speakers), "number_of_units": len(plan.units)})

## 13. Azure source-embedding download

In [ ]:
sensitive_paths = set()
azure_asset_keys = set()
source_cache_pairs = 0
if WORKER_MODE == "assets":
    for source in plan.sources:
        remote = remote_sources[source.azure_voice]
        for field in ("calibration_audio_path", "calibration_manifest_path"):
            local_path = Path(getattr(source, field))
            remote_path = remote[field]
            identity = (remote_path, local_path)
            if identity in azure_asset_keys:
                continue
            blob = store.bucket.blob(store.name(JOB_ID, remote_path))
            if not blob.exists():
                raise FileNotFoundError("A planned Azure calibration asset is missing")
            blob.reload()
            if not blob.size or int(blob.size) > MAX_DOWNLOAD_BYTES:
                raise RuntimeError("A planned Azure calibration asset violates the download limit")
            store.download(JOB_ID, remote_path, local_path)
            azure_asset_keys.add(identity)
            sensitive_paths.add(local_path)
        embedding_blob = store.bucket.blob(store.name(JOB_ID, remote["source_embedding_path"]))
        embedding_manifest_blob = store.bucket.blob(store.name(JOB_ID, remote["source_embedding_manifest_path"]))
        embedding_exists = embedding_blob.exists()
        embedding_manifest_exists = embedding_manifest_blob.exists()
        if embedding_exists != embedding_manifest_exists:
            raise RuntimeError("Promoted Azure source cache must contain both embedding and manifest")
        if embedding_exists:
            for field, blob in (
                ("source_embedding_path", embedding_blob),
                ("source_embedding_manifest_path", embedding_manifest_blob),
            ):
                blob.reload()
                if not blob.size or int(blob.size) > MAX_DOWNLOAD_BYTES:
                    raise RuntimeError("A promoted Azure source cache object violates the download limit")
                local_path = Path(getattr(source, field))
                remote_path = remote[field]
                store.download(JOB_ID, remote_path, local_path)
                azure_asset_keys.add((remote_path, local_path))
                sensitive_paths.add(local_path)
            source_cache_pairs += 1
else:
    for unit in plan.units:
        remote = remote_units[unit.unit_id]
        for field in ("source_audio_path", "source_embedding_path"):
            local_path = Path(getattr(unit, field))
            remote_path = remote[field]
            identity = (remote_path, local_path)
            if identity in azure_asset_keys:
                continue
            blob = store.bucket.blob(store.name(JOB_ID, remote_path))
            if not blob.exists():
                raise FileNotFoundError("A planned Azure source asset is missing")
            blob.reload()
            if not blob.size or int(blob.size) > MAX_DOWNLOAD_BYTES:
                raise RuntimeError("A planned Azure source asset violates the download limit")
            store.download(JOB_ID, remote_path, local_path)
            azure_asset_keys.add(identity)
            sensitive_paths.add(local_path)
print({"worker_mode": WORKER_MODE, "azure_source_assets": len(azure_asset_keys), "source_cache_pairs": source_cache_pairs})

## 14. Speaker reference and target-embedding download

In [ ]:
reference_assets = {}
target_assets = set()
candidate_input_assets = set()
if WORKER_MODE == "assets":
    for speaker in plan.speakers:
        remote = remote_speakers[speaker.speaker_id]
        for field, expected_hash in (
            ("reference_audio_path", speaker.reference_audio_sha256),
            ("reference_manifest_path", speaker.reference_manifest_sha256),
        ):
            local_path = Path(getattr(speaker, field))
            relative = remote[field]
            blob = store.bucket.blob(store.name(JOB_ID, relative))
            if not blob.exists():
                raise FileNotFoundError(f"The same-speaker reference asset is missing for {speaker.speaker_id}")
            blob.reload()
            if not blob.size or int(blob.size) > MAX_DOWNLOAD_BYTES:
                raise RuntimeError(f"The same-speaker reference asset violates the download limit for {speaker.speaker_id}")
            store.download(JOB_ID, relative, local_path)
            reference_assets[(speaker.speaker_id, field)] = (local_path, expected_hash)
            sensitive_paths.add(local_path)
        for candidate in speaker.voice_candidates:
            relative = remote["voice_candidates"][candidate.candidate_id]["source_audio_path"]
            local_path = candidate.source_audio_path
            blob = store.bucket.blob(store.name(JOB_ID, relative))
            if not blob.exists():
                raise FileNotFoundError("A planned Azure speaker-calibration candidate is missing")
            blob.reload()
            if not blob.size or int(blob.size) > MAX_DOWNLOAD_BYTES:
                raise RuntimeError("A planned Azure speaker-calibration candidate violates the download limit")
            store.download(JOB_ID, relative, local_path)
            candidate_input_assets.add((relative, local_path))
            sensitive_paths.add(local_path)
else:
    for speaker_id in speakers:
        spec = speaker_reference_specs[speaker_id]
        relative = job_relative(spec.get("path", ""))
        expected_hash = str(spec.get("sha256", ""))
        if not SHA256_RE.fullmatch(expected_hash):
            raise ValueError("Every speaker reference requires an authoritative SHA-256")
        local_path = JOB_ROOT / relative
        blob = store.bucket.blob(store.name(JOB_ID, relative))
        if not blob.exists():
            raise FileNotFoundError(f"The same-speaker reference is missing for {speaker_id}")
        blob.reload()
        if not blob.size or int(blob.size) > MAX_DOWNLOAD_BYTES:
            raise RuntimeError(f"The same-speaker reference violates the download limit for {speaker_id}")
        store.download(JOB_ID, relative, local_path)
        reference_assets[speaker_id] = (local_path, expected_hash)
        sensitive_paths.add(local_path)
    for unit in plan.units:
        local_path = unit.target_embedding_path
        relative = remote_units[unit.unit_id]["target_embedding_path"]
        identity = (relative, local_path)
        if identity in target_assets:
            continue
        blob = store.bucket.blob(store.name(JOB_ID, relative))
        if not blob.exists():
            raise FileNotFoundError("A planned target-speaker embedding is missing")
        blob.reload()
        if not blob.size or int(blob.size) > MAX_DOWNLOAD_BYTES:
            raise RuntimeError("A planned target-speaker embedding violates the download limit")
        store.download(JOB_ID, relative, local_path)
        target_assets.add(identity)
        sensitive_paths.add(local_path)
print({"worker_mode": WORKER_MODE, "speaker_reference_assets": len(reference_assets), "target_embeddings": len(target_assets), "voice_candidate_inputs": len(candidate_input_assets)})

## 15. Input hash validation

In [ ]:
from mathula_tv.quality import inspect_wav

validated_inputs = set()
if WORKER_MODE == "assets":
    for source in plan.sources:
        for label, path, expected_hash in (
            ("Azure calibration audio", source.calibration_audio_path, source.calibration_audio_sha256),
            ("Azure calibration manifest", source.calibration_manifest_path, source.calibration_manifest_sha256),
        ):
            if sha256_file(path) != expected_hash:
                raise RuntimeError(f"Planned input hash mismatch: {label}")
            validated_inputs.add((str(path), expected_hash))
        if not inspect_wav(source.calibration_audio_path).get("valid"):
            raise RuntimeError("Azure calibration audio is not a valid PCM WAV")
        embedding_exists = source.source_embedding_path.is_file()
        embedding_manifest_exists = source.source_embedding_manifest_path.is_file()
        if embedding_exists != embedding_manifest_exists:
            raise RuntimeError("Local Azure source cache must contain both embedding and manifest")
    for speaker in plan.speakers:
        for label, path, expected_hash in (
            ("speaker reference audio", speaker.reference_audio_path, speaker.reference_audio_sha256),
            ("speaker reference manifest", speaker.reference_manifest_path, speaker.reference_manifest_sha256),
        ):
            if sha256_file(path) != expected_hash:
                raise RuntimeError(f"Planned input hash mismatch: {label}")
            validated_inputs.add((str(path), expected_hash))
        if not inspect_wav(speaker.reference_audio_path).get("valid"):
            raise RuntimeError("Same-speaker reference is not a valid PCM WAV")
        for candidate in speaker.voice_candidates:
            if sha256_file(candidate.source_audio_path) != candidate.source_audio_sha256 or not inspect_wav(candidate.source_audio_path).get("valid"):
                raise RuntimeError("Azure voice candidate input validation failed")
            validated_inputs.add((str(candidate.source_audio_path), candidate.source_audio_sha256))
else:
    for unit in plan.units:
        for label, path, expected_hash in (
            ("Azure source audio", unit.source_audio_path, unit.source_audio_sha256),
            ("Azure source embedding", unit.source_embedding_path, unit.source_embedding_sha256),
            ("target-speaker embedding", unit.target_embedding_path, unit.target_embedding_sha256),
        ):
            if sha256_file(path) != expected_hash:
                raise RuntimeError(f"Planned input hash mismatch: {label}")
            validated_inputs.add((str(path), expected_hash))
        source_quality = inspect_wav(unit.source_audio_path)
        if not source_quality.get("valid") or source_quality.get("channels") != 1 or source_quality.get("sample_rate") != unit.sample_rate:
            raise RuntimeError("Azure source audio is not canonical mono PCM WAV")
    for speaker_id, (path, expected_hash) in reference_assets.items():
        if sha256_file(path) != expected_hash or not inspect_wav(path).get("valid"):
            raise RuntimeError(f"Same-speaker reference validation failed for {speaker_id}")
        validated_inputs.add((str(path), expected_hash))
print({"worker_mode": WORKER_MODE, "input_hash_validation": "verified", "validated_inputs": len(validated_inputs)})

## 16. Model load

In [ ]:
import importlib

from mathula_tv.openvoice import OpenVoiceBackend, OpenVoicePins

factory_module, separator, factory_name = RUNTIME_FACTORY.partition(":")
if not separator or not factory_module or not factory_name or not all(SAFE_ID_RE.fullmatch(part) for part in factory_module.split(".")) or not SAFE_ID_RE.fullmatch(factory_name):
    raise ValueError("MATHULA_TV_OPENVOICE_RUNTIME_FACTORY must be a safe module:function reference")
runtime_factory = getattr(importlib.import_module(factory_module), factory_name)
runtime = runtime_factory()
candidate_evaluator = None
if WORKER_MODE == "assets" and CANDIDATE_EVALUATOR_FACTORY:
    evaluator_module, evaluator_separator, evaluator_name = CANDIDATE_EVALUATOR_FACTORY.partition(":")
    if not evaluator_separator or not evaluator_module or not evaluator_name or not all(SAFE_ID_RE.fullmatch(part) for part in evaluator_module.split(".")) or not SAFE_ID_RE.fullmatch(evaluator_name):
        raise ValueError("MATHULA_TV_OPENVOICE_CANDIDATE_EVALUATOR_FACTORY must be a safe module:function reference")
    candidate_evaluator = getattr(importlib.import_module(evaluator_module), evaluator_name)()
    if not callable(candidate_evaluator):
        raise TypeError("The candidate evaluator factory must return a callable")
pins = OpenVoicePins(revision=OPENVOICE_REVISION, python_version=EXPECTED_PYTHON_VERSION, torch_version=EXPECTED_TORCH_VERSION, checkpoints=tuple(checkpoint_assets))
backend = OpenVoiceBackend(runtime, pins, device="cuda", precision=PRECISION)
runtime_info = backend.load()
print({"worker_mode": WORKER_MODE, "model_loaded": True, "gpu": runtime_info.gpu_name, "cuda_version": runtime_info.cuda_version, "python_version": runtime_info.python_version, "torch_version": runtime_info.torch_version, "openvoice_revision": runtime_info.openvoice_revision, "checkpoint_hashes": backend.checkpoint_hashes})

## 17. Pending-turn conversion

In [ ]:
from mathula_tv.openvoice_worker import OpenVoiceAssetWorker, OpenVoiceWorker

manifest_path = JOB_ROOT / MANIFEST_RELATIVE
with GCSLeaseHeartbeat(store, claim, LEASE_SECONDS) as lease_heartbeat:
    if WORKER_MODE == "assets":
        worker = OpenVoiceAssetWorker(backend, plan, manifest_path, lease_guard=lease_heartbeat.assert_owned, max_attempts_per_asset=MAX_ATTEMPTS, allow_lease_rebind=True, candidate_evaluator=candidate_evaluator)
        pending_assets = worker.pending_asset_ids()
        for asset_id in pending_assets:
            print({"current_asset": asset_id})
        manifest = worker.run()
    else:
        worker = OpenVoiceWorker(backend, plan, manifest_path, lease_guard=lease_heartbeat.assert_owned, max_attempts_per_unit=MAX_ATTEMPTS, allow_lease_rebind=True)
        pending_units = worker.pending_unit_ids()
        units_by_id = {unit.unit_id: unit for unit in plan.units}
        for unit_id in pending_units:
            print({"current_unit": unit_id, "selected_azure_voice": units_by_id[unit_id].azure_voice})
        manifest = worker.run()
claim = lease_heartbeat.claim
if WORKER_MODE == "assets":
    print({"worker_mode": WORKER_MODE, "completed_assets": manifest["completed_assets"], "pending_assets": manifest["pending_assets"]})
else:
    print({"worker_mode": WORKER_MODE, "completed_units": manifest["completed_units"], "pending_units": manifest["pending_units"]})

## 18. WAV validation

In [ ]:
promotion_assets = []
if WORKER_MODE == "assets":
    for source in plan.sources:
        result = manifest["sources"][source.azure_voice].get("result", {})
        if sha256_file(source.source_embedding_path) != result.get("source_embedding_sha256") or sha256_file(source.source_embedding_manifest_path) != result.get("source_embedding_manifest_sha256"):
            raise RuntimeError(f"OpenVoice source embedding output validation failed for {source.azure_voice}")
        sensitive_paths.add(source.source_embedding_path)
        promotion_assets.append((f"source:{source.azure_voice}", remote_sources[source.azure_voice]["source_embedding_path"], source.source_embedding_path, result["source_embedding_sha256"]))
        promotion_assets.append((f"source-manifest:{source.azure_voice}", remote_sources[source.azure_voice]["source_embedding_manifest_path"], source.source_embedding_manifest_path, result["source_embedding_manifest_sha256"]))
        print({"current_asset": f"source:{source.azure_voice}", "output_sha256": result["source_embedding_sha256"]})
    for speaker in plan.speakers:
        speaker_record = manifest["speakers"][speaker.speaker_id]
        result = speaker_record.get("result", {})
        if sha256_file(speaker.target_embedding_path) != result.get("target_embedding_sha256") or sha256_file(speaker.target_embedding_manifest_path) != result.get("target_embedding_manifest_sha256"):
            raise RuntimeError(f"OpenVoice target embedding output validation failed for {speaker.speaker_id}")
        sensitive_paths.add(speaker.target_embedding_path)
        promotion_assets.append((f"speaker:{speaker.speaker_id}", remote_speakers[speaker.speaker_id]["target_embedding_path"], speaker.target_embedding_path, result["target_embedding_sha256"]))
        promotion_assets.append((f"speaker-manifest:{speaker.speaker_id}", remote_speakers[speaker.speaker_id]["target_embedding_manifest_path"], speaker.target_embedding_manifest_path, result["target_embedding_manifest_sha256"]))
        print({"current_asset": f"speaker:{speaker.speaker_id}", "output_sha256": result["target_embedding_sha256"]})
        for candidate in speaker.voice_candidates:
            candidate_result = speaker_record["candidates"][candidate.candidate_id].get("result", {})
            quality = inspect_wav(candidate.output_path)
            if sha256_file(candidate.output_path) != candidate_result.get("output_sha256") or not quality.get("valid") or quality.get("channels") != 1 or quality.get("sample_rate") != candidate.sample_rate:
                raise RuntimeError(f"OpenVoice voice-candidate WAV validation failed for {candidate.candidate_id}")
            sensitive_paths.add(candidate.output_path)
            relative = remote_speakers[speaker.speaker_id]["voice_candidates"][candidate.candidate_id]["output_path"]
            promotion_assets.append((f"candidate:{candidate.candidate_id}", relative, candidate.output_path, candidate_result["output_sha256"]))
            print({"current_asset": f"candidate:{candidate.candidate_id}", "selected_azure_voice": candidate.azure_voice, "source_duration_ms": candidate_result["source_duration_ms"], "converted_duration_ms": candidate_result["converted_duration_ms"], "conversion_ratio": candidate_result["duration_ratio"], "output_sha256": candidate_result["output_sha256"]})
else:
    for unit in plan.units:
        record = manifest["units"][unit.unit_id]
        result = record.get("result", {})
        if record.get("status") != "completed" or sha256_file(unit.output_path) != result.get("output_sha256"):
            raise RuntimeError(f"OpenVoice output is incomplete or has an invalid hash for {unit.unit_id}")
        quality = inspect_wav(unit.output_path)
        if not quality.get("valid") or quality.get("channels") != 1 or quality.get("sample_rate") != unit.sample_rate:
            raise RuntimeError(f"OpenVoice output WAV validation failed for {unit.unit_id}")
        sensitive_paths.add(unit.output_path)
        promotion_assets.append((unit.unit_id, remote_units[unit.unit_id]["output_path"], unit.output_path, result["output_sha256"]))
        print({"current_unit": unit.unit_id, "selected_azure_voice": unit.azure_voice, "source_duration_ms": result["source_duration_ms"], "converted_duration_ms": result["converted_duration_ms"], "conversion_ratio": result["duration_ratio"], "output_sha256": result["output_sha256"]})

## 19. GCS promotion

In [ ]:
for asset_id, final_relative, local_path, expected_hash in promotion_assets:
    claim = store.heartbeat(claim, LEASE_SECONDS)
    store.assert_claim(claim)
    temporary_relative = f"{final_relative}.partial.{WORKER_ID}"
    temporary_blob = store.bucket.blob(store.name(JOB_ID, temporary_relative))
    if not temporary_blob.exists():
        store.upload(JOB_ID, temporary_relative, local_path, if_generation_match=0)
    store.promote(JOB_ID, temporary_relative, final_relative, expected_hash, sha256_file)
    store.assert_claim(claim)
    if WORKER_MODE == "assets":
        print({"current_asset": asset_id, "output_sha256": expected_hash, "promotion_result": "verified"})
    else:
        print({"current_unit": asset_id, "output_sha256": expected_hash, "promotion_result": "verified"})

## 20. Manifest creation

In [ ]:
from mathula_tv.atomic_io import atomic_write_json

claim = store.heartbeat(claim, LEASE_SECONDS)
store.assert_claim(claim)
pending_count = manifest.get("pending_assets") if WORKER_MODE == "assets" else manifest.get("pending_units")
if manifest.get("state") != "completed" or pending_count != 0:
    raise RuntimeError("The worker manifest is not complete and cannot be promoted as final")
manifest["server_plan_file_sha256"] = PLAN_SHA256
if WORKER_MODE == "assets":
    server_plan_artifact_sha256 = str(plan_payload.get("artifact_sha256", ""))
    if not SHA256_RE.fullmatch(server_plan_artifact_sha256):
        raise RuntimeError("The server OpenVoice asset plan lacks a valid artifact identity")
    manifest["server_plan_artifact_sha256"] = server_plan_artifact_sha256
    for speaker in plan.speakers:
        if speaker.candidate_manifest_path is None:
            continue
        relative = remote_speakers[speaker.speaker_id]["candidate_manifest_path"]
        candidate_manifest_sha256 = sha256_file(speaker.candidate_manifest_path)
        temporary = f"{relative}.partial.{WORKER_ID}"
        temporary_blob = store.bucket.blob(store.name(JOB_ID, temporary))
        if not temporary_blob.exists():
            store.upload(JOB_ID, temporary, speaker.candidate_manifest_path, if_generation_match=0)
        store.promote(JOB_ID, temporary, relative, candidate_manifest_sha256, sha256_file)
        store.assert_claim(claim)
        print({"manifest_path": relative, "manifest_sha256": candidate_manifest_sha256, "promotion_result": "verified"})
atomic_write_json(manifest_path, manifest)
manifest_sha256 = sha256_file(manifest_path)
manifest_temporary = f"{MANIFEST_RELATIVE}.partial.{WORKER_ID}"
manifest_temporary_blob = store.bucket.blob(store.name(JOB_ID, manifest_temporary))
if not manifest_temporary_blob.exists():
    store.upload(JOB_ID, manifest_temporary, manifest_path, if_generation_match=0)
store.promote(JOB_ID, manifest_temporary, MANIFEST_RELATIVE, manifest_sha256, sha256_file)
store.assert_claim(claim)
print({"worker_mode": WORKER_MODE, "manifest_path": MANIFEST_RELATIVE, "manifest_sha256": manifest_sha256, "promotion_result": "verified"})

## 21. Lease completion

In [ ]:
store.assert_claim(claim)
claim = store.finish_claim(claim, "completed")
print({"job_id": JOB_ID, "lease_owner": claim["worker_id"], "lease_status": claim["status"]})

## 22. Server-reconciliation instructions

In [ ]:
next_subcommand = "reconcile-openvoice-assets" if WORKER_MODE == "assets" else "reconcile-voice-conversion"
next_command = f"python -m mathula_tv.cli {next_subcommand} {JOB_ID} --live-operation"
print({"worker_mode": WORKER_MODE, "job_id": JOB_ID, "manifest_path": MANIFEST_RELATIVE, "next_command": next_command})

## 23. Local sensitive-file cleanup

In [ ]:
deleted_files = 0
for sensitive_path in sorted(sensitive_paths, key=str):
    if sensitive_path.is_file():
        sensitive_path.unlink()
        deleted_files += 1
store = None
gcs_client = None
kaggle_credentials = None
print({"local_sensitive_cleanup": "completed", "deleted_files": deleted_files})